<a href="https://colab.research.google.com/github/K-K-0/MLP-Assignments/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
import numpy as np
import pandas as pd


In [61]:
df = pd.read_csv('/content/processed_data.csv')
df.head()

,text,sentiment,Time of Tweet,Age of User,Land Area (Km²),Continent,Density_Level,Population_Group
0,Sooo SAD I will miss you here in San Diego!!!,negative,noon,21-30,27400.0,EU,Medium,Medium
1,my boss is bullying me...,negative,night,31-45,2381740.0,AF,Low,Medium
2,what interview! leave me alone,negative,morning,46-60,470.0,EU,Medium,Low
3,"Sons of ****, why couldn`t they put them on t...",negative,noon,60-70,1246700.0,AF,Low,Medium
4,2am feedings for the baby are fun when he is a...,positive,morning,0-20,2736690.0,SA,Low,Medium


In [62]:
from sklearn.model_selection import train_test_split
X = df.drop('sentiment', axis=1)
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=0)
X_train.head()

,text,Time of Tweet,Age of User,Land Area (Km²),Continent,Density_Level,Population_Group
2554,"come home, then. Not so boring here.",morning,0-20,62710.0,AS,Medium,Medium
5412,Is feeling like he has a bad flu. Yes. Bad. Flu.,morning,46-60,25680.0,AF,Medium,Medium
7503,have a lok at EF too! they are jummy,morning,46-60,365000.0,EU,Low,Medium
8694,"PINKPOP weekend & I`ve got NO tickets, meaning...",night,31-45,60.0,EU,High,Low
194,seriously bored without anyone to talk to... b...,morning,46-60,24670.0,AF,High,Medium


In [63]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['Land Area (Km²)']),
    ('ordinal', OrdinalEncoder(categories=[['Low', 'Medium', 'High'],['Low', 'Medium', 'High']]), ['Density_Level', 'Population_Group']),
    ('Onehot', OneHotEncoder(drop='first', sparse_output=False),['Time of Tweet',	'Age of User', 'Continent']),
    ('tfidf', TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b|[@#]\w+',
    strip_accents='unicode'
), 'text')
], remainder='passthrough')

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

In [68]:
print(X_train.columns)

Index(['text', 'Time of Tweet', 'Age of User', 'Land Area (Km²)', 'Continent',
       'Density_Level', 'Population_Group'],
      dtype='object')


In [65]:
np.sum(X_test_trans[:5])

np.float64(26.885773869406624)

In [71]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

preprocessor = ColumnTransformer([

    # Ordinal
    ('ordinal', OrdinalEncoder(categories=[
        ['Low', 'Medium', 'High'],
        ['Low', 'Medium', 'High']
    ]), ['Density_Level', 'Population_Group']),

    # OneHot
    ('onehot', OneHotEncoder(drop='first', sparse_output=False),
     ['Time of Tweet', 'Age of User', 'Continent']),

    # Text
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        max_features=5000,
        ngram_range=(1, 2),
        token_pattern=r'(?u)\b\w\w+\b|[@#]\w+',
        strip_accents='unicode'
    ), 'text')
])

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_transformed, y_train)

from sklearn.metrics import log_loss

y_prob = model.predict_proba(X_test_transformed)

loss = log_loss(y_test, y_prob)

round(loss, 4)

0.3687

In [74]:
from sklearn.ensemble import RandomForestClassifier

# Use full preprocessor (including Land Area)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

model = RandomForestClassifier(random_state=42)
model.fit(X_train_transformed, y_train)

y_pred = model.predict(X_test_transformed)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("Positive" if cm[0,1] > cm[1,0] else "Negative")

Negative


In [75]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

rfecv = RFECV(
    estimator=LogisticRegression(random_state=42, max_iter=1000),
    step=100,
    n_jobs=-1
)

rfecv.fit(X_train_transformed, y_train)

print("Selected features:", rfecv.n_features_)

Selected features: 3915


In [76]:
import zipfile
import os

zip_path = "/content/archive.zip"   # change to your zip file name
extract_path = "dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [77]:
import numpy as np
import cv2
import os

In [83]:
images = []
labels = []

dataset_path = "/content/dataset/data"   # folder after unzip

for label in os.listdir(dataset_path):
    label_path = os.path.join(dataset_path, label)

    for img in os.listdir(label_path):
        img_path = os.path.join(label_path, img)

        # Read image
        image = cv2.imread(img_path)
        if image is None:
            continue
        # Convert to grayscale
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Resize to 100x100
        image = cv2.resize(image, (100, 100))

        # Normalize
        image = image / 255.0

        # Flatten
        image = image.flatten()

        images.append(image)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)
print(images)

[[0.7372549  0.72941176 0.7254902  ... 0.05882353 0.0627451  0.06666667]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.99607843 1.         1.         ... 1.         1.         1.        ]
 ...
 [0.99607843 0.99607843 0.99607843 ... 0.45882353 0.50196078 0.37254902]
 [0.2745098  0.28235294 0.30196078 ... 0.05098039 0.05098039 0.05098039]
 [0.12941176 0.1372549  0.14509804 ... 0.50196078 0.49019608 0.4627451 ]]


In [84]:
np.sum(labels == "without_mask")

np.int64(3828)

In [85]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

labels_encoded = le.fit_transform(labels)

# Fix mapping manually
# Ensure without_mask = 1 and with_mask = 0
mapping = {'with_mask': 0, 'without_mask': 1}
labels_encoded = np.array([mapping[label] for label in labels])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels_encoded,
    test_size=0.2,
    random_state=0
)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=0,
    max_iter=500,
    tol=0.001,
    C=10
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

import numpy as np

false_positive = np.sum((y_pred == 1) & (y_test == 0))

print(false_positive)

286


In [86]:
np.random.seed(0)

augmentation_factor = 2

angle_of_rotation = np.random.uniform(
    -180, 180,
    size=(augmentation_factor * len(X_train),)
)

In [87]:
import numpy as np
import cv2

def augment_image(images, labels, angles, augmentation_factor):

    augmented_images = []
    augmented_labels = []

    idx = 0

    for i in range(len(images)):

        # Add original image
        original = images[i].reshape(100, 100)
        augmented_images.append(original)
        augmented_labels.append(labels[i])

        # Create augmented images
        for _ in range(augmentation_factor):

            angle = angles[idx]
            idx += 1

            # Rotation matrix
            M = cv2.getRotationMatrix2D((50, 50), angle, 1)

            # Rotate image
            rotated = cv2.warpAffine(original, M, (100, 100))

            augmented_images.append(rotated)
            augmented_labels.append(labels[i])

    return np.array(augmented_images), np.array(augmented_labels)

In [88]:
augmented_images, augmented_labels = augment_image(
    X_train,
    y_train,
    angle_of_rotation,
    augmentation_factor=2
)

In [89]:
sum_value = np.sum(augmented_labels[7000:8000])

print(sum_value)

485


In [91]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=0)

rf.fit(augmented_images.reshape(len(augmented_images), -1), augmented_labels)

RandomForestClassifier(random_state=0)

In [92]:
import numpy as np

importances = rf.feature_importances_

top_100_features = np.argsort(importances)[-100:]

X_train_selected = augmented_images.reshape(len(augmented_images), -1)[:, top_100_features]

X_test_selected = X_test.reshape(len(X_test), -1)[:, top_100_features]

rf_selected = RandomForestClassifier(random_state=0)

rf_selected.fit(X_train_selected, augmented_labels)

y_pred = rf_selected.predict(X_test_selected)

misclassified = np.sum(y_pred != y_test)

print(misclassified)

270


In [93]:
X_train_aug = augmented_images.reshape(len(augmented_images), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

from sklearn.decomposition import PCA

pca = PCA(n_components=100, random_state=0)

X_train_pca = pca.fit_transform(X_train_aug)
X_test_pca = pca.transform(X_test_flat)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=0)

rf.fit(X_train_pca, augmented_labels)

y_pred = rf.predict(X_test_pca)

In [94]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(accuracy)

0.7954996690933157
